# Non-GP probabilistic surrogates\n\nRandom Forest / Extra Trees / bootstrap boosting を同じ BoTorch posterior contract で扱う最小例です。ensemble のばらつきはモデルごとに意味が異なるため、calibrated Gaussian posterior としては解釈しません。

In [ ]:
import torch\nfrom robotorchan.models import RandomForestSurrogate\n\ntrain_X = torch.linspace(0, 1, 20, dtype=torch.double).unsqueeze(-1)\ntrain_Y = torch.sin(2 * torch.pi * train_X)\nmodel = RandomForestSurrogate(train_X, train_Y, n_estimators=32, random_state=0)\nmodel.fit()

In [ ]:
test_X = torch.linspace(0, 1, 8, dtype=torch.double).unsqueeze(-1)\nposterior = model.posterior(test_X)\nposterior.mean.shape, posterior.variance.shape

## MC acquisition\n\nNon-GP ensemble posterior は sampleable なので MC acquisition と接続できます。sklearn tree の候補入力に対する勾配は利用できないため、最適化には gradient-free search を使います。

In [ ]:
from botorch.acquisition.monte_carlo import qExpectedImprovement\n\nacqf = qExpectedImprovement(model=model, best_f=train_Y.max())\nvalue = acqf(torch.tensor([[[0.35]]], dtype=torch.double))\nvalue

## Boosting の注意\n\nGradient Boosting / HistGradientBoosting では boosting stage 自体を posterior sample にしません。robotorchan は bootstrap ごとに complete boosting model を学習し、その complete-model predictions を empirical posterior members とします。